# Chapter 01: 数据清洗

本 Notebook 对应 `src/ch01_data_cleaning/preprocess.py`，以交互式方式执行数据清洗流程。

**清洗步骤：**
- Step 1.1: 数据加载与结构探查
- Step 1.2: 缺失值检测与处理
- Step 1.3: 重复值检测与处理
- Step 1.4: 数据类型验证与转换
- Step 1.5: 异常值检测与处理
- Step 1.6: 清洗后数据保存与报告生成

In [ ]:
import sys
from pathlib import Path

# 项目根目录
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import (
    RAW_DATA_FILE, OUTPUT_BASE, COLUMN_CONFIG,
    ENTITY_CONFIG, DOMAIN_PARAMS, CATEGORY_MAP, PLT_STYLE,
)
from src.utils.data_loader import load_raw_data
from src.utils.output_manager import save_dataframe, save_figure, save_markdown, ensure_dir
from src.utils.visualizer import plot_heatmap
from src.utils.task_graph import TaskGraph

print(f"项目根目录: {PROJECT_ROOT}")
print(f"原始数据: {RAW_DATA_FILE}")

---
## Step 1.1: 数据加载与结构探查

In [ ]:
# 加载原始数据
df_raw = load_raw_data()
df = df_raw.copy()

In [ ]:
# 数据形状
print(f"数据形状: {df.shape[0]} 行 x {df.shape[1]} 列")
print(f"\n列名: {list(df.columns)}")

In [ ]:
# 数据类型概览
print(df.dtypes)

In [ ]:
# 前 5 行预览
df.head()

In [ ]:
# 数值列描述统计
df.describe()

In [ ]:
# 品牌分布
df["brand"].value_counts().plot(kind="barh", figsize=(10, 8))
plt.title("品牌分布")
plt.xlabel("数量")
plt.tight_layout()
plt.show()

In [ ]:
# 年份分布
df["year"].value_counts().sort_index().plot(kind="bar")
plt.title("年份分布")
plt.xlabel("年份")
plt.ylabel("数量")
plt.tight_layout()
plt.show()

---
## Step 1.2: 缺失值检测与处理

In [ ]:
# 缺失值统计
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({"缺失数量": null_counts, "缺失比例(%)": null_pct})
null_summary[null_summary["缺失数量"] > 0]

In [ ]:
# TODO: 根据业务需求补充缺失值处理逻辑
# 示例:
# df['column_name'].fillna(df['column_name'].median(), inplace=True)
print("缺失值处理: 当前数据无缺失值，跳过。")

---
## Step 1.3: 重复值检测与处理

In [ ]:
# 完全重复检测
n_full_dup = df.duplicated().sum()
print(f"完全重复记录数: {n_full_dup}")

In [ ]:
# 业务键重复检测（品牌+型号+年份+变体）
business_key = ["brand", "model", "year", "variant"]
n_key_dup = df.duplicated(subset=business_key).sum()
print(f"业务键重复记录数: {n_key_dup}")

In [ ]:
# TODO: 根据检测结果执行去重操作
# df = df.drop_duplicates()
print(f"当前数据形状: {df.shape}")

---
## Step 1.4: 数据类型验证与转换

In [ ]:
# 数据类型验证
expected_types = {
    "brand": "object", "model": "object", "year": "int64",
    "variant": "object", "price_usd": "float64",
    "battery_capacity_kwh": "float64", "range_miles": "float64",
    "charging_speed_kw": "float64", "acceleration_0_60_mph": "float64",
    "top_speed_mph": "float64", "horsepower": "float64",
    "torque_nm": "float64", "drive_type": "object",
    "seating_capacity": "int64", "body_type": "object",
    "cargo_volume_cubic_ft": "float64", "weight_kg": "float64",
    "safety_rating": "int64", "autopilot_level": "int64",
    "country_of_origin": "object", "market_segment": "object",
    "annual_sales_units": "int64", "customer_rating": "float64",
    "warranty_years": "int64",
}

for col, expected in expected_types.items():
    actual = str(df[col].dtype)
    status = "OK" if actual == expected else f"MISMATCH"
    print(f"  {col:30s} -> {actual:15s} [{status}]")

In [ ]:
# 将分类列转换为 category 类型
categorical_cols = COLUMN_CONFIG["categorical_columns"]
for col in categorical_cols:
    df[col] = df[col].astype("category")

print(f"分类列已转换: {categorical_cols}")
print(f"内存占用: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

---
## Step 1.5: 异常值检测与处理

In [ ]:
# IQR 方法检测异常值
numeric_cols = COLUMN_CONFIG["numeric_columns"]
outlier_results = []

for col in numeric_cols:
    if col not in df.columns:
        continue
    series = df[col].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((series < lower) | (series > upper)).sum()
    if n_out > 0:
        outlier_results.append({
            "列名": col, "Q1": round(q1, 2), "Q3": round(q3, 2),
            "IQR": round(iqr, 2), "下界": round(lower, 2),
            "上界": round(upper, 2), "异常值数量": n_out,
        })

outlier_df = pd.DataFrame(outlier_results)
if not outlier_df.empty:
    print(outlier_df.to_string(index=False))
else:
    print("未检测到异常值。")

In [ ]:
# TODO: 根据业务需求处理异常值
# 示例（裁剪）:
# for col in [...]:
#     q1, q3 = df[col].quantile([0.25, 0.75])
#     iqr = q3 - q1
#     df[col] = df[col].clip(q1 - 1.5*iqr, q3 + 1.5*iqr)
print("异常值处理: 请根据业务需求补充处理逻辑。")

---
## Step 1.6: 清洗后数据保存与报告生成

In [ ]:
# 保存清洗后的数据
output_dir = OUTPUT_BASE / "ch01_data_cleaning"
ensure_dir(output_dir)

cleaned_path = save_dataframe(df, output_dir / "cleaned_data.csv")
print(f"清洗后数据: {cleaned_path}")

In [ ]:
# 清洗后数据预览
df.head(10)

In [ ]:
# 最终数据形状
print(f"最终数据形状: {df.shape[0]} 行 x {df.shape[1]} 列")
print(f"\n数据清洗流程完成!")